# V3 Pipeline Training — Optimized Feature Engineering + Grid Search

**Degisiklikler:**
- Meta-predictor prediction label'lari **LabelEncode** edilip feature olarak kullaniliyor
- Ayri one-hot encoding (48 feature: ref_base, alt_base, ref_amino, alt_amino)
- Manuel k-mer encoding (DNA tam kombinatorik, protein gozlenen vocab)
- Secilmis FE: Grantham, BLOSUM62, conservation interaction, is_transition
- Grid Search (Optuna yerine) — 12 combo/model, 3-fold CV
- 5-fold CV degerlendirme
- Otomatik grafikler ve PDF rapor

In [8]:
# Cell 1: Imports & Config
import sys, os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Proje root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, DATA_PATH, RESULTS_V2_DIR, MODELS_V2_DIR, REPORTS_DIR, PANELS_SINGLE
from src.features import prepare_data_v3
from src.metrics import optimize_threshold, compute_all_metrics
from src.models import grid_search_lightgbm, grid_search_xgboost, grid_search_nn, GRID_BUILDERS
from src.utils import prepare_for_xgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

# Sonuc klasoru
V3_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v3')
V3_MODELS_DIR = os.path.join(PROJECT_ROOT, 'models', 'v3')
os.makedirs(V3_RESULTS_DIR, exist_ok=True)
os.makedirs(V3_MODELS_DIR, exist_ok=True)

print(f"Proje root: {PROJECT_ROOT}")
print(f"Veri: {DATA_PATH}")
print(f"Sonuclar: {V3_RESULTS_DIR}")

Proje root: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Veri: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\data\raw\open_cravat_curation_v3.csv
Sonuclar: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v3


In [9]:
# Cell 2: Data Loading & Feature Engineering + Hold-Out Split
print("=" * 60)
print("ADIM 1: Veri yukleme")
print("=" * 60)

df_raw = pd.read_csv(DATA_PATH)
print(f"Ham veri: {df_raw.shape[0]} satir x {df_raw.shape[1]} sutun")

# Target derivation
label_map = {
    'Pathogenic': 1, 'Likely pathogenic': 1,
    'Benign': 0, 'Likely benign': 0,
}
df_raw['target'] = df_raw['clinvar__sig'].map(label_map)
df_raw = df_raw.dropna(subset=['target'])
df_raw['target'] = df_raw['target'].astype(int)
print(f"Target: Pozitif={df_raw['target'].sum()}, Negatif={(df_raw['target']==0).sum()}")

# Panel bilgisini ayir
panel_series = df_raw['Panel'].copy()

print("\n" + "=" * 60)
print("ADIM 2: V3 Feature Engineering")
print("=" * 60)

df_v3 = prepare_data_v3(df_raw)

# Panel'i geri ekle (drop edilmis olabilir)
if 'Panel' not in df_v3.columns:
    df_v3['Panel'] = panel_series.values

print(f"\nFinal feature sayisi: {df_v3.shape[1] - 2} (target + Panel haric)")
print(f"Ornekler: {df_v3.shape[0]}")

# ---- Hold-Out Split: %80 CV, %20 Test ----
print("\n" + "=" * 60)
print("ADIM 2b: Hold-Out Split (%80 CV / %20 Test)")
print("=" * 60)

X_full = df_v3.drop(columns=['target', 'Panel'])
y_full = df_v3['target']

X_cv, X_holdout, y_cv, y_holdout = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=y_full
)

print(f"CV seti:      {X_cv.shape[0]} ornek ({X_cv.shape[0]/len(y_full)*100:.1f}%)")
print(f"Hold-out set: {X_holdout.shape[0]} ornek ({X_holdout.shape[0]/len(y_full)*100:.1f}%)")
print(f"CV target dagilimi:      Pozitif={y_cv.sum()}, Negatif={(y_cv==0).sum()}")
print(f"Hold-out target dagilimi: Pozitif={y_holdout.sum()}, Negatif={(y_holdout==0).sum()}")


ADIM 1: Veri yukleme
Ham veri: 4287 satir x 119 sutun
Target: Pozitif=2951, Negatif=1336

ADIM 2: V3 Feature Engineering
  LabelEncode esm1b__prediction: 2 sinif
  LabelEncode metalr__pred: 2 sinif
  LabelEncode metarnn__pred: 2 sinif
  LabelEncode metasvm__pred: 2 sinif
  LabelEncode mistic__pred: 2 sinif
  LabelEncode mutationtaster__prediction: 4 sinif
  LabelEncode phdsnpg__prediction: 2 sinif
  LabelEncode provean__prediction: 2 sinif
  LabelEncode sift__prediction: 2 sinif
  LabelEncode alphamissense__am_class: 3 sinif
  OHE (ayri alfabe): 48 feature
  K-mer DNA_11mer_Ref: 16 2-mer feature
  K-mer DNA_11mer_Alt: 16 2-mer feature
  K-mer Prot_11mer_Ref: 430 2-mer feature
  K-mer Prot_11mer_Alt: 434 2-mer feature
  57 gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 15 sutun dusuruldu
[V3] Veri boyutu: 4287 x 991

Final feature sayisi: 989 (target + Panel haric)
Ornekler: 4287

ADIM 2b: Hold-Out Split (%80 CV / %20 Test)
CV seti:      3429 ornek (80.0%)
Hold-out set: 858 ornek 

In [ ]:
# Cell 3: 5-Fold CV Training (on %80) + Hold-Out Test (on %20)
import time
import copy
import joblib

print("=" * 60)
print("ADIM 3: 5-Fold CV Egitim (Grid Search) — sadece CV seti uzerinde")
print("=" * 60)

# Kategorik feature'lari tespit et
cat_features = [c for c in X_cv.columns if X_cv[c].dtype == 'category']

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

model_types = ['lightgbm', 'xgboost']
all_results = []
fold_details = {mt: {'y_true': [], 'y_prob': [], 'y_pred': [], 'f1': []} for mt in model_types}
best_models = {}  # Her model tipi icin en iyi fold modeli (en yuksek F1)
best_f1_per_mt = {mt: -1.0 for mt in model_types}

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_cv, y_cv)):
    print(f"\n{'='*40} FOLD {fold_idx+1}/{N_FOLDS} {'='*40}")
    X_train_fold = X_cv.iloc[train_idx].copy()
    y_train_fold = y_cv.iloc[train_idx].copy()
    X_test_fold = X_cv.iloc[test_idx].copy()
    y_test_fold = y_cv.iloc[test_idx].copy()

    for mt in model_types:
        print(f"\n--- {mt.upper()} ---")
        t0 = time.time()

        builder = GRID_BUILDERS[mt]
        model, best_combo, best_thr, y_prob = builder(
            X_train_fold, y_train_fold, X_test_fold, y_test_fold, cat_features
        )
        elapsed = time.time() - t0

        y_pred = (y_prob >= best_thr).astype(int)
        metrics = compute_all_metrics(y_test_fold, y_pred, y_prob)

        # Kaydet
        fold_details[mt]['y_true'].extend(y_test_fold.values)
        fold_details[mt]['y_prob'].extend(y_prob)
        fold_details[mt]['y_pred'].extend(y_pred)
        fold_details[mt]['f1'].append(metrics['f1'])

        result_row = {
            'fold': fold_idx + 1,
            'model': mt,
            'panel': 'ALL',
            'fe_mode': 'v3',
            'split': 'cv',
            'threshold': best_thr,
            **metrics,
            'elapsed_sec': round(elapsed, 1),
            'best_params': str(best_combo),
        }
        all_results.append(result_row)

        print(f"  F1={metrics['f1']:.4f}  AUC-ROC={metrics['auc_roc']:.4f}  "
              f"Precision={metrics['precision']:.4f}  Recall={metrics['recall']:.4f}  "
              f"({elapsed:.1f}s)")

        # En iyi F1'li fold modelini sakla
        if metrics['f1'] > best_f1_per_mt[mt]:
            best_f1_per_mt[mt] = metrics['f1']
            best_models[mt] = {
                'model': model,
                'threshold': best_thr,
                'fold': fold_idx + 1,
                'best_params': best_combo,
                'cv_f1': metrics['f1'],
            }

# ---- CV Ozet ----
print("\n" + "=" * 60)
print("5-FOLD CV OZET (sadece CV seti)")
print("=" * 60)
for mt in model_types:
    f1s = fold_details[mt]['f1']
    print(f"  {mt:12s}: F1 = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}  "
          f"(min={np.min(f1s):.4f}, max={np.max(f1s):.4f})")



ADIM 3: 5-Fold CV Egitim (Grid Search) — sadece CV seti uzerinde

======================================== FOLD 1/5 ========================================

--- LIGHTGBM ---
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.1} -> CV F1=0.9842
  F1=0.9862  AUC-ROC=0.9950  Precision=0.9852  Recall=0.9873  (34.1s)

--- XGBOOST ---
  XGBoost Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1} -> CV F1=0.9849
  F1=0.9852  AUC-ROC=0.9958  Precision=0.9811  Recall=0.9894  (42.3s)

======================================== FOLD 2/5 ========================================

--- LIGHTGBM ---
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.1} -> CV F1=0.9841
  F1=0.9820  AUC-ROC=0.9962  Precision=0.9810  Recall=0.9831  (34.6s)

--- XGBOOST ---
  XGBoost Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 20

TypeError: Not supported type for data.<class 'xgboost.core.DMatrix'>

In [12]:
# ---- Hold-Out Test ----
print("\n" + "=" * 60)
print("HOLD-OUT TEST (%20 — hic gorulmemis veri)")
print("=" * 60)

holdout_results = {}
for mt in model_types:
    bm = best_models[mt]
    model = bm['model']

    # Hold-out uzerinde tahmin
    X_ho = X_holdout.copy()
    if mt == 'xgboost':
        for col in X_ho.select_dtypes(include=['category']).columns:
            X_ho[col] = X_ho[col].cat.codes
    y_prob_ho = model.predict_proba(X_ho)[:, 1]

    # Ayni threshold'u kullan
    thr = bm['threshold']
    y_pred_ho = (y_prob_ho >= thr).astype(int)
    metrics_ho = compute_all_metrics(y_holdout, y_pred_ho, y_prob_ho)

    holdout_results[mt] = {
        'y_true': y_holdout,
        'y_prob': y_prob_ho,
        'y_pred': y_pred_ho,
        'threshold': thr,
        'metrics': metrics_ho,
        'model': model,
    }

    result_row = {
        'fold': 'holdout',
        'model': mt,
        'panel': 'ALL',
        'fe_mode': 'v3',
        'split': 'holdout',
        'threshold': thr,
        **metrics_ho,
        'elapsed_sec': 0,
        'best_params': str(bm['best_params']),
    }
    all_results.append(result_row)

    cv_f1 = np.mean(fold_details[mt]['f1'])
    print(f"  {mt:12s}: Hold-Out F1={metrics_ho['f1']:.4f}  (CV F1={cv_f1:.4f})")
    print(f"               AUC-ROC={metrics_ho['auc_roc']:.4f}  Precision={metrics_ho['precision']:.4f}  "
          f"Recall={metrics_ho['recall']:.4f}")
    print(f"               Threshold={thr:.3f}  (Fold {bm['fold']}'den)")

# Sonuclari kaydet
results_df = pd.DataFrame(all_results)
results_df.to_csv(os.path.join(V3_RESULTS_DIR, 'v3_cv_results.csv'), index=False)
print(f"\nSonuclar kaydedildi: {os.path.join(V3_RESULTS_DIR, 'v3_cv_results.csv')}")



HOLD-OUT TEST (%20 — hic gorulmemis veri)
  lightgbm    : Hold-Out F1=0.9891  (CV F1=0.9841)
               AUC-ROC=0.9989  Precision=0.9833  Recall=0.9949
               Threshold=0.140  (Fold 1'den)
  xgboost     : Hold-Out F1=0.9899  (CV F1=0.9850)
               AUC-ROC=0.9987  Precision=0.9882  Recall=0.9915
               Threshold=0.560  (Fold 4'den)

Sonuclar kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v3\v3_cv_results.csv


In [13]:
# Cell 4: Otomatik Grafikler (CV + Hold-Out)
print("=" * 60)
print("ADIM 4: Grafikler")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('V3 Pipeline — 5-Fold CV + Hold-Out Test Sonuclari', fontsize=16, fontweight='bold')

colors = {'lightgbm': '#2196F3', 'xgboost': '#FF9800'}

# --- 1. ROC Curve (CV) ---
ax = axes[0, 0]
for mt in model_types:
    y_true_all = np.array(fold_details[mt]['y_true'])
    y_prob_all = np.array(fold_details[mt]['y_prob'])
    fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
    auc_val = roc_auc_score(y_true_all, y_prob_all)
    ax.plot(fpr, tpr, color=colors[mt], lw=2, label=f'{mt} CV (AUC={auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve (CV)')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# --- 2. ROC Curve (Hold-Out) ---
ax = axes[0, 1]
for mt in model_types:
    hr = holdout_results[mt]
    fpr, tpr, _ = roc_curve(hr['y_true'], hr['y_prob'])
    auc_val = roc_auc_score(hr['y_true'], hr['y_prob'])
    ax.plot(fpr, tpr, color=colors[mt], lw=2, label=f'{mt} (AUC={auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve (Hold-Out %20)')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)

# --- 3. PR Curve (Hold-Out) ---
ax = axes[0, 2]
for mt in model_types:
    hr = holdout_results[mt]
    prec, rec, _ = precision_recall_curve(hr['y_true'], hr['y_prob'])
    ap = average_precision_score(hr['y_true'], hr['y_prob'])
    ax.plot(rec, prec, color=colors[mt], lw=2, label=f'{mt} (AP={ap:.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve (Hold-Out %20)')
ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)

# --- 4. CM Hold-Out LightGBM ---
ax = axes[1, 0]
hr = holdout_results['lightgbm']
cm = confusion_matrix(hr['y_true'], hr['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('LightGBM CM (Hold-Out)')

# --- 5. CM Hold-Out XGBoost ---
ax = axes[1, 1]
hr = holdout_results['xgboost']
cm = confusion_matrix(hr['y_true'], hr['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('XGBoost CM (Hold-Out)')

# --- 6. F1 Box Plot + Hold-Out yildiz ---
ax = axes[1, 2]
f1_data = [fold_details[mt]['f1'] for mt in model_types]
bp = ax.boxplot(f1_data, labels=[mt.upper() for mt in model_types], patch_artist=True)
for patch, mt in zip(bp['boxes'], model_types):
    patch.set_facecolor(colors[mt]); patch.set_alpha(0.7)
for i, mt in enumerate(model_types):
    ax.scatter([i+1]*len(fold_details[mt]['f1']), fold_details[mt]['f1'],
               color='black', s=40, zorder=3, alpha=0.7, label='CV fold' if i == 0 else None)
    ho_f1 = holdout_results[mt]['metrics']['f1']
    ax.scatter(i+1, ho_f1, color='red', s=150, zorder=4, marker='*',
               label='Hold-Out' if i == 0 else None)
ax.set_ylabel('F1 Score')
ax.set_title('F1 Dagilimi: CV Folds + Hold-Out')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(rect=[0, 0, 1, 0.95])
graphs_path = os.path.join(V3_RESULTS_DIR, 'v3_graphs.png')
fig.savefig(graphs_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafikler kaydedildi: {graphs_path}")


ADIM 4: Grafikler
Grafikler kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v3\v3_graphs.png


In [14]:
# Cell 5: Otomatik PDF Rapor (CV + Hold-Out)
from datetime import datetime

pdf_path = os.path.join(REPORTS_DIR, f'v3_report_{datetime.now().strftime("%Y%m%d_%H%M")}.pdf')

# Feature kategori renkleri
def _feat_color(name):
    n = name.lower()
    if '__score' in n or '__phred' in n or 'pathogenicity' in n: return '#E53935'
    if '__prediction' in n or '__pred' in n or '__am_class' in n: return '#AB47BC'
    if 'gerp' in n or 'phylop' in n or 'phastcons' in n or 'conservation' in n: return '#43A047'
    if 'gnomad' in n or 'log10_af' in n or 'af_bin' in n or 'homozygote' in n: return '#FF9800'
    if 'blosum' in n or 'grantham' in n: return '#1E88E5'
    if 'mer_' in n or 'kmer' in n: return '#78909C'
    if 'delta_' in n or 'polarity' in n or 'chirality' in n: return '#00ACC1'
    if '__ref_base' in n or '__alt_base' in n or 'ref_amino' in n or 'alt_amino' in n: return '#FDD835'
    return '#90A4AE'

with PdfPages(pdf_path) as pdf:
    # --- Sayfa 1: Kapak + Ozet ---
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.5, 0.88, "V3 Pipeline — Egitim Raporu", fontsize=24, fontweight='bold',
            ha='center', va='top', transform=ax.transAxes)
    ax.text(0.5, 0.82, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
            fontsize=12, ha='center', va='top', transform=ax.transAxes, color='gray')

    lines = [
        "PIPELINE OZELLIKLERI:",
        f"  Feature Engineering: V3 (Label-encoded predictions + ayri OHE + manuel k-mer)",
        f"  Feature sayisi: {X_cv.shape[1]}",
        f"  Toplam ornek: {len(y_full)}  (CV: {len(y_cv)}, Hold-Out: {len(y_holdout)})",
        f"  Model turleri: LightGBM, XGBoost",
        f"  Hyperparameter: Grid Search (12 combo/model, 3-fold CV)",
        f"  Degerlendirme: 5-Fold Stratified CV + %20 Hold-Out Test",
        "",
        "5-FOLD CV SONUCLARI (egitim seti %80):",
    ]
    for mt in model_types:
        f1s = fold_details[mt]['f1']
        lines.append(f"  {mt:12s}: F1 = {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
    lines += ["", "HOLD-OUT TEST SONUCLARI (hic gorulmemis %20):"]
    for mt in model_types:
        hm = holdout_results[mt]['metrics']
        cv_f1 = np.mean(fold_details[mt]['f1'])
        lines.append(f"  {mt:12s}: F1={hm['f1']:.4f}  AUC={hm['auc_roc']:.4f}  "
                      f"(CV F1={cv_f1:.4f})")
    lines += ["", "KARSILASTIRMA (full_fe vs v3 hold-out):"]
    prev_path = os.path.join(RESULTS_V2_DIR, 'full_fe_results.csv')
    if os.path.exists(prev_path):
        prev_df = pd.read_csv(prev_path)
        for mt in model_types:
            prev_row = prev_df[(prev_df['model_type'] == mt) & (prev_df['panel'] == 'All')]
            if len(prev_row) > 0:
                prev_f1 = prev_row['f1'].values[0]
                ho_f1 = holdout_results[mt]['metrics']['f1']
                delta = ho_f1 - prev_f1
                lines.append(f"  {mt}: full_fe F1={prev_f1:.4f} -> v3 Hold-Out F1={ho_f1:.4f} ({delta:+.4f})")
    else:
        lines.append("  (full_fe_results.csv bulunamadi)")

    ax.text(0.08, 0.68, '\n'.join(lines), fontsize=10.5, fontfamily='monospace',
            va='top', transform=ax.transAxes)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # --- Sayfa 2: ROC/PR/CM (Hold-Out odakli) ---
    fig2, axes2 = plt.subplots(2, 2, figsize=(16, 12))
    fig2.suptitle('Hold-Out Test Sonuclari (%20 — Hic Gorulmemis Veri)', fontsize=16, fontweight='bold')

    # ROC Hold-Out
    ax = axes2[0, 0]
    for mt in model_types:
        hr = holdout_results[mt]
        fpr, tpr, _ = roc_curve(hr['y_true'], hr['y_prob'])
        auc_val = roc_auc_score(hr['y_true'], hr['y_prob'])
        ax.plot(fpr, tpr, color=colors[mt], lw=2, label=f'{mt} (AUC={auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve (Hold-Out)')
    ax.legend(); ax.grid(True, alpha=0.3)

    # PR Hold-Out
    ax = axes2[0, 1]
    for mt in model_types:
        hr = holdout_results[mt]
        prec, rec, _ = precision_recall_curve(hr['y_true'], hr['y_prob'])
        ap = average_precision_score(hr['y_true'], hr['y_prob'])
        ax.plot(rec, prec, color=colors[mt], lw=2, label=f'{mt} (AP={ap:.4f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title('PR Curve (Hold-Out)')
    ax.legend(); ax.grid(True, alpha=0.3)

    # CM Hold-Out LightGBM
    ax = axes2[1, 0]
    hr = holdout_results['lightgbm']
    cm = confusion_matrix(hr['y_true'], hr['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title('LightGBM CM (Hold-Out)')

    # CM Hold-Out XGBoost
    ax = axes2[1, 1]
    hr = holdout_results['xgboost']
    cm = confusion_matrix(hr['y_true'], hr['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax, cmap='Oranges', colorbar=False)
    ax.set_title('XGBoost CM (Hold-Out)')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig2, bbox_inches='tight'); plt.close(fig2)

    # --- Sayfa 3: Feature Importance (tam sayfa) ---
    fig3, (ax_lgbm, ax_xgb) = plt.subplots(1, 2, figsize=(22, 14))
    fig3.suptitle('Feature Importance — Top 30 (En Iyi CV Fold Modeli)', fontsize=18, fontweight='bold')
    feature_names = X_cv.columns
    TOP_N = 30

    if 'lightgbm' in best_models:
        imp = best_models['lightgbm']['model'].feature_importances_
        top_idx = np.argsort(imp)[-TOP_N:]
        top_names = [feature_names[i] for i in top_idx]
        top_colors = [_feat_color(n) for n in top_names]
        ax_lgbm.barh(range(TOP_N), imp[top_idx], color=top_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
        ax_lgbm.set_yticks(range(TOP_N))
        ax_lgbm.set_yticklabels(top_names, fontsize=8)
        ax_lgbm.set_xlabel('Importance (split count)', fontsize=11)
        ax_lgbm.set_title('LightGBM', fontsize=14, fontweight='bold')
        ax_lgbm.grid(True, alpha=0.3, axis='x')

    if 'xgboost' in best_models:
        imp = best_models['xgboost']['model'].feature_importances_
        top_idx = np.argsort(imp)[-TOP_N:]
        top_names = [feature_names[i] if i < len(feature_names) else f'f{i}' for i in top_idx]
        top_colors = [_feat_color(n) for n in top_names]
        ax_xgb.barh(range(TOP_N), imp[top_idx], color=top_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
        ax_xgb.set_yticks(range(TOP_N))
        ax_xgb.set_yticklabels(top_names, fontsize=8)
        ax_xgb.set_xlabel('Importance (gain)', fontsize=11)
        ax_xgb.set_title('XGBoost', fontsize=14, fontweight='bold')
        ax_xgb.grid(True, alpha=0.3, axis='x')

    from matplotlib.patches import Patch
    legend_items = [
        Patch(facecolor='#E53935', label='In-silico skor'),
        Patch(facecolor='#AB47BC', label='Prediction label'),
        Patch(facecolor='#43A047', label='Conservation'),
        Patch(facecolor='#FF9800', label='gnomAD'),
        Patch(facecolor='#1E88E5', label='Substitution (BLOSUM/Grantham)'),
        Patch(facecolor='#78909C', label='K-mer'),
        Patch(facecolor='#00ACC1', label='Physicochemical'),
        Patch(facecolor='#FDD835', label='OHE (base/amino)'),
    ]
    fig3.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=10,
                frameon=True, fancybox=True, shadow=True, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.04, 1, 0.95])
    pdf.savefig(fig3, bbox_inches='tight'); plt.close(fig3)

    # --- Sayfa 4: CV vs Hold-Out Karsilastirma ---
    fig4, (ax_box, ax_bar) = plt.subplots(1, 2, figsize=(16, 8))
    fig4.suptitle('CV vs Hold-Out Karsilastirma', fontsize=16, fontweight='bold')

    # F1 box plot + hold-out yildiz
    f1_data = [fold_details[mt]['f1'] for mt in model_types]
    bp = ax_box.boxplot(f1_data, labels=[mt.upper() for mt in model_types], patch_artist=True)
    for patch, mt in zip(bp['boxes'], model_types):
        patch.set_facecolor(colors[mt]); patch.set_alpha(0.7)
    for i, mt in enumerate(model_types):
        ax_box.scatter([i+1]*len(fold_details[mt]['f1']), fold_details[mt]['f1'],
                       color='black', s=50, zorder=3, label='CV fold' if i == 0 else None)
        ho_f1 = holdout_results[mt]['metrics']['f1']
        ax_box.scatter(i+1, ho_f1, color='red', s=200, zorder=4, marker='*',
                       label='Hold-Out' if i == 0 else None)
    ax_box.set_ylabel('F1 Score', fontsize=12)
    ax_box.set_title('F1: CV Folds vs Hold-Out', fontsize=13)
    ax_box.legend(loc='lower right')
    ax_box.grid(True, alpha=0.3, axis='y')

    # Metrik karsilastirma: CV ort vs Hold-Out
    metric_names = ['f1', 'auc_roc', 'auc_pr', 'precision', 'recall']
    metric_labels = ['F1', 'AUC-ROC', 'AUC-PR', 'Precision', 'Recall']
    x = np.arange(len(metric_names))
    width = 0.2
    cv_rows = results_df[results_df['split'] == 'cv']
    for i, mt in enumerate(model_types):
        mt_cv = cv_rows[cv_rows['model'] == mt]
        cv_means = [mt_cv[m].mean() for m in metric_names]
        ho_vals = [holdout_results[mt]['metrics'][m] for m in metric_names]
        ax_bar.bar(x + i*2*width, cv_means, width, label=f'{mt} CV', color=colors[mt], alpha=0.6)
        ax_bar.bar(x + i*2*width + width, ho_vals, width, label=f'{mt} HO', color=colors[mt], alpha=1.0,
                   edgecolor='black', linewidth=1.2)
    ax_bar.set_xticks(x + 1.5*width)
    ax_bar.set_xticklabels(metric_labels, fontsize=10)
    ax_bar.set_ylabel('Score', fontsize=12)
    ax_bar.set_title('CV Ortalama vs Hold-Out', fontsize=13)
    ax_bar.legend(fontsize=9)
    ax_bar.grid(True, alpha=0.3, axis='y')
    ax_bar.set_ylim(0, 1.05)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig4, bbox_inches='tight'); plt.close(fig4)

    # --- Sayfa 5: Detayli sonuc tablosu (CV + Hold-Out) ---
    fig5, ax5 = plt.subplots(figsize=(15, 9))
    ax5.axis('off')
    ax5.set_title('Detayli Sonuclar: 5-Fold CV + Hold-Out Test', fontsize=14, fontweight='bold', pad=20)

    table_data = []
    cv_rows_sorted = results_df[results_df['split'] == 'cv'].sort_values(['model', 'fold'])
    for _, row in cv_rows_sorted.iterrows():
        table_data.append([
            int(row['fold']), row['model'], 'CV',
            f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}", f"{row['auc_pr']:.4f}",
            f"{row['precision']:.4f}", f"{row['recall']:.4f}", f"{row['mcc']:.4f}",
        ])
    # CV ortalamalar
    for mt in model_types:
        mt_cv = cv_rows[cv_rows['model'] == mt]
        table_data.append([
            'ORT', mt, 'CV',
            f"{mt_cv['f1'].mean():.4f}", f"{mt_cv['auc_roc'].mean():.4f}",
            f"{mt_cv['auc_pr'].mean():.4f}", f"{mt_cv['precision'].mean():.4f}",
            f"{mt_cv['recall'].mean():.4f}", f"{mt_cv['mcc'].mean():.4f}",
        ])
    # Hold-out
    for mt in model_types:
        hm = holdout_results[mt]['metrics']
        table_data.append([
            '-', mt, 'HOLD-OUT',
            f"{hm['f1']:.4f}", f"{hm['auc_roc']:.4f}", f"{hm['auc_pr']:.4f}",
            f"{hm['precision']:.4f}", f"{hm['recall']:.4f}", f"{hm['mcc']:.4f}",
        ])

    col_labels = ['Fold', 'Model', 'Split', 'F1', 'AUC-ROC', 'AUC-PR', 'Precision', 'Recall', 'MCC']
    table = ax5.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.4)
    # Header
    for j in range(len(col_labels)):
        table[0, j].set_facecolor('#4CAF50')
        table[0, j].set_text_props(color='white', fontweight='bold')
    # CV ortalama satirlari (yesil)
    n_cv = len(cv_rows_sorted)
    for i in range(len(model_types)):
        row_idx = n_cv + i + 1
        for j in range(len(col_labels)):
            table[row_idx, j].set_facecolor('#E8F5E9')
            table[row_idx, j].set_text_props(fontweight='bold')
    # Hold-out satirlari (turuncu)
    for i in range(len(model_types)):
        row_idx = n_cv + len(model_types) + i + 1
        for j in range(len(col_labels)):
            table[row_idx, j].set_facecolor('#FFF3E0')
            table[row_idx, j].set_text_props(fontweight='bold')

    pdf.savefig(fig5, bbox_inches='tight'); plt.close(fig5)

print(f"\nPDF rapor kaydedildi: {pdf_path}")
print(f"  5 sayfa: Kapak/Ozet, Hold-Out Grafikleri, Feature Importance, CV vs Hold-Out, Detayli Tablo")



PDF rapor kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\v3_report_20260322_2031.pdf
  5 sayfa: Kapak/Ozet, Hold-Out Grafikleri, Feature Importance, CV vs Hold-Out, Detayli Tablo


In [ ]:
# Cell 6 (Opsiyonel): NN Grid Search Egitimi
# Bu cell uzun surer (~30-60 dk). Calistirmak icin comment'i kaldir.
# LightGBM/XGBoost sonuclarini gormek icin bu cell'i atlayabilirsiniz.

"""
print("=" * 60)
print("ADIM 5: NN Grid Search Egitimi")
print("=" * 60)

nn_results = []
nn_fold_f1s = []

skf_nn = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold_idx, (train_idx, test_idx) in enumerate(skf_nn.split(X_all, y_all)):
    print(f"\\nFOLD {fold_idx+1}/5 - Neural Network")
    X_train_fold = X_all.iloc[train_idx].copy()
    y_train_fold = y_all.iloc[train_idx].copy()
    X_test_fold = X_all.iloc[test_idx].copy()
    y_test_fold = y_all.iloc[test_idx].copy()

    t0 = time.time()
    model, best_combo, best_thr, y_prob = grid_search_nn(
        X_train_fold, y_train_fold, X_test_fold, y_test_fold, cat_features
    )
    elapsed = time.time() - t0

    y_pred = (y_prob >= best_thr).astype(int)
    metrics = compute_all_metrics(y_test_fold, y_pred, y_prob)
    nn_fold_f1s.append(metrics['f1'])

    nn_results.append({
        'fold': fold_idx + 1, 'model': 'nn', 'panel': 'ALL', 'fe_mode': 'v3',
        'threshold': best_thr, **metrics, 'elapsed_sec': round(elapsed, 1),
        'best_params': str(best_combo),
    })
    print(f"  F1={metrics['f1']:.4f}  AUC-ROC={metrics['auc_roc']:.4f}  ({elapsed:.1f}s)")

nn_df = pd.DataFrame(nn_results)
# Mevcut sonuclara ekle
combined_df = pd.concat([results_df, nn_df], ignore_index=True)
combined_df.to_csv(os.path.join(V3_RESULTS_DIR, 'v3_cv_results.csv'), index=False)
print(f"\\nNN 5-fold CV: F1 = {np.mean(nn_fold_f1s):.4f} +/- {np.std(nn_fold_f1s):.4f}")
"""
print("NN egitimi devre disi. Calistirmak icin docstring'i kaldir.")